# Classification

Classification predicts a category. This notebook moves from a binary question — does a person earn
more than 50k? — through four different classifiers, then shows a multi-class problem with the
iris measurements. Along the way it demonstrates encoding categorical features properly.

## Learning objectives

By the end of this notebook you will be able to:

- encode categorical features with one-hot encoding inside a pipeline;
- split data with stratification so class balance is preserved;
- fit logistic regression, a decision tree, an SVM, and KNN;
- evaluate with accuracy, precision, recall, and F1;
- explain why accuracy alone is misleading on a balanced-looking but skewed target.

## Concept

In **classification** the target is a label. `Logistic regression` models the probability of the
positive class and is linear in the log-odds; it is fast and interpretable. A **decision tree**
splits the feature space into rectangles and is easy to explain but prone to overfitting. An
**SVM** finds a boundary that maximises the margin, using a kernel to bend it. **KNN** predicts
from the labels of the nearest training points and needs scaled features so distances are fair.

Categorical features such as `workclass` cannot be fed to these algorithms as text. **One-hot
encoding** turns each category into its own 0/1 column. Doing this inside a `ColumnTransformer`
keeps preprocessing with the model, so the test set is transformed with the exact same categories
seen in training.

**Metrics:** accuracy is the share correct. Precision answers "of those predicted positive, how
many are right?"; recall answers "of the actual positives, how many did we catch?"; F1 balances
the two. With a skewed target, accuracy can be high while recall for the minority class is
terrible, so report all four.

## Worked example

### Load and inspect

The Adult table is large, so we take a reproducible subsample to keep the demonstration quick.

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
import ml
from ds_practice import load_uci_adult, set_seed, classification_metrics

set_seed(42)
adult = load_uci_adult()
print("full shape:", adult.shape)
print("income balance:\n", adult["income"].value_counts(normalize=True).round(3).to_string())

adult = adult.sample(6000, random_state=42).reset_index(drop=True)
print("\nsample shape:", adult.shape)

full shape: (32561, 15)
income balance:
 income
<=50K    0.759
>50K     0.241

sample shape: (6000, 15)


### Encode the target and split

We map the target to 0/1 and stratify the split so the class ratio is preserved.

In [2]:
from sklearn.model_selection import train_test_split

adult = adult.assign(target=(adult["income"] == ">50K").astype(int))
y = adult["target"]
X = adult.drop(columns=["income", "target"])

numeric = X.select_dtypes("number").columns.tolist()
categorical = X.select_dtypes("object").columns.tolist()
print("numeric:", numeric)
print("categorical:", categorical)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("train/test:", len(X_train), len(X_test))

numeric: ['age', 'fnlwgt', 'education_num', 'capital_gain', 'capital_loss', 'hours_per_week']
categorical: ['workclass', 'education', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'native_country']
train/test: 4800 1200


### A preprocessing pipeline

`ColumnTransformer` applies one-hot encoding to the categorical columns and passes the numeric ones
through. `handle_unknown="ignore"` prevents a category seen only in the test set from raising.

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

preprocess = ColumnTransformer([
    ("num", "passthrough", numeric),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical),
])

### Four classifiers

Each model is wrapped with the preprocessing (and `ml.make_classifier` adds feature scaling).

In [4]:
models = {
    "logistic": ml.make_classifier("logistic"),
    "tree": ml.make_classifier("tree", max_depth=6),
    "svm": ml.make_classifier("svm"),
    "knn": ml.make_classifier("knn", n_neighbors=15),
}
rows = []
for name, model in models.items():
    pipeline = Pipeline([("prep", preprocess), ("model", model)])
    pipeline.fit(X_train, y_train)
    metrics = classification_metrics(y_test, pipeline.predict(X_test))
    rows.append({"model": name, **{k: round(v, 3) for k, v in metrics.items()}})
display(pd.DataFrame(rows))

,model,accuracy,precision,recall,f1
0,logistic,0.859,0.853,0.859,0.853
1,tree,0.856,0.854,0.856,0.842
2,svm,0.851,0.844,0.851,0.841
3,knn,0.828,0.820,0.828,0.822


### More than two classes

The iris dataset has three species. Logistic regression extends naturally to multinomial
classification; the confusion matrix shows which species get mixed up.

In [5]:
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

iris = load_iris(as_frame=True)
Xi_train, Xi_test, yi_train, yi_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42, stratify=iris.target
)
multi = ml.make_classifier("logistic").fit(Xi_train, yi_train)
print("accuracy:", round(multi.score(Xi_test, yi_test), 3))
display(pd.DataFrame(
    confusion_matrix(yi_test, multi.predict(Xi_test)),
    index=list(iris.target_names), columns=list(iris.target_names),
))

accuracy: 0.933


,setosa,versicolor,virginica
setosa,10,0,0
versicolor,0,9,1
virginica,0,1,9


## Exercises

1. **Class weighting.** Refit logistic regression with `class_weight="balanced"` and compare recall
   on the `>50K` class with the default. Explain the trade-off with precision.
2. **Scaling matters.** Fit KNN with `n_neighbors=5` on unscaled and scaled features and compare
   accuracy. Why does scaling change KNN so much?
3. **Tree depth.** Sweep `max_depth` over 2, 6, and 12 for the tree classifier and report how train
   and test accuracy diverge.

## Limitations

The Adult sample is a snapshot and its `fnlwgt` column is a sampling weight, not a feature, which
we leave in for simplicity and which a careful analysis would handle. Accuracy hides the cost of
different mistakes; in an income-screening context a false positive and a false negative are not
equally bad. SVM training is slow on large data, which is why the sample is small. Finally, the
one-hot representation loses any ordinal information in categories such as education level.